# The numbers your computer cannot store

Every number and every character in your data is encoded in bits. The
lecture explained three examples: two's complement for signed
integers, IEEE 754 for real numbers, and UTF-8 for text. This notebook shows what can go wrong 
with those encodings in ordinary data work in three small
experiments.

## 1. Rounding errors add up

$0.1$ has no finite binary expansion, so the computer stores a number that is
very slightly off. One such error is harmless, but a million of them are not. Here we
add $0.1$ a million times, first in the usual way and then with `math.fsum`,
which keeps track of the lost digits.

In [ ]:
import math

total = 0.0
for _ in range(1_000_000):     # First time you see this? This is a loop where the looping variable doesn't matter
    total += 0.1
    
print("plain loop:", total)
print("math.fsum: ", math.fsum([0.1] * 1_000_000))
print("error of the loop:", total - 100_000)

plain loop: 100000.00000133288
math.fsum:  100000.0
error of the loop: 1.3328826753422618e-06


The loop is off by about $10^{-6}$, but it will be noticeable in a formula you may use every week. The variance can
be computed in one pass as the mean of the squares minus the square of the mean.
On paper that is exact. In floating point it subtracts two huge, nearly equal
numbers, and the digits that survive are the ones that were wrong due to the encoding. Here
the data are Unix timestamps (about $1.7 \cdot 10^9$ seconds) of events that
happened within a couple of seconds of each other, so the true variance is
about 1.

In [ ]:
rng = np.random.default_rng(1)
timestamps = 1_700_000_000 + rng.normal(0, 1.0, size=10_000)   # seconds since 1970, which is the standard for timestamps

naive = np.mean(timestamps ** 2) - np.mean(timestamps) ** 2      
two_pass = np.mean((timestamps - np.mean(timestamps)) ** 2)      
print("one-pass formula:", naive)
print("two-pass formula:", two_pass)
print("numpy.var:       ", np.var(timestamps))

: 

The one-pass formula returns exactly 0 because the variance of the data should have been encoded in
digits that a 64-bit float does not have once the numbers are squared. With another seed the same formula comes out negative, which is an impossible value for a variance. Luckily, the two-pass formula and `numpy.var` agree.

## 2. An integer that becomes negative

Python integers grow without bound, but the arrays of NumPy and pandas store
numbers in fixed-width types, exactly the `byte`, `short`, `int` and `long` of
the lecture's table. A 32-bit integer stops at $2^{31} - 1 = 2\,147\,483\,647$.
Timestamps in seconds fit, but converting them to milliseconds does not, and
NumPy does not stop to tell you.

In [4]:
seconds = np.array([1_700_000_000, 1_700_000_001], dtype=np.int32)   # seconds since 1970
millis = seconds * 1000                                                # still int32
print("as int32:     ", millis)
print("Python ints:  ", [int(s) * 1000 for s in seconds])
print("as int64:     ", seconds.astype(np.int64) * 1000)

as int32:      [-807049216 -807048216]
Python ints:   [1700000000000, 1700000001000]
as int64:      [1700000000000 1700000001000]


The products wrap around and become large negative numbers, without an error
and without a warning because the operation is done element by element in
32-bit two's complement. Anything you compute from that column afterwards is
quietly wrong. Check the `dtype` of a column when its values are near a power of
two, and convert to a wider type before you multiply.

## 3. One byte, two names

Text is bits too, and the map from characters to bytes is not unique. `ß` and
`ü` are one byte each in Latin-1 (ISO-8859-1) and two bytes each in UTF-8, so
the same file means different things under the two encodings.

In [5]:
for ch in "ß", "ü", "a":
    print(f"{ch}: UTF-8 {ch.encode('utf-8').hex(' ')!s:8}  Latin-1 {ch.encode('latin-1').hex(' ')}")

ß: UTF-8 c3 9f     Latin-1 df
ü: UTF-8 c3 bc     Latin-1 fc
a: UTF-8 61        Latin-1 61


Now the everyday version of the problem. A colleague saves a survey as a
UTF-8 file, which is what nearly every program does today. We read it with
`encoding="latin-1"`, which is what many older tools and some spreadsheets
still assume.

In [6]:
import io
import pandas as pd

csv_bytes = "name,age\nMüller,34\nMueller,41\nMüller,29\nGroß,52\n".encode("utf-8")

right = pd.read_csv(io.BytesIO(csv_bytes), encoding="utf-8")
wrong = pd.read_csv(io.BytesIO(csv_bytes), encoding="latin-1")
print(wrong)
print("\nrows named Müller:", (right["name"] == "Müller").sum(), "read correctly,",
      (wrong["name"] == "Müller").sum(), "with the wrong encoding")
print("mean age of the Müllers:", right.loc[right["name"] == "Müller", "age"].mean(),
      " and with the wrong encoding:", wrong.loc[wrong["name"] == "Müller", "age"].mean())

      name  age
0  MÃ¼ller   34
1  Mueller   41
2  MÃ¼ller   29
3    GroÃ   52

rows named Müller: 2 read correctly, 0 with the wrong encoding
mean age of the Müllers: 31.5  and with the wrong encoding: nan


Each two-byte UTF-8 character has become two Latin-1 characters: `Ã¼` for
`ü`, and for `ß` an `Ã` followed by an invisible control character but nothing
crashed. The file still has the right number of
rows and columns; only every name with a non-ASCII letter is now a different
string, so a group of respondents disappears from every filter and every group
mean that names it. The fix is one argument, `encoding="utf-8"`, but you have
to know that the question exists.

**What we used:** [floating-point numbers](../Transcripts/02-InformationCoding.md#7-floating-point-numbers-ieee-754),
[when rounding errors matter](../Transcripts/02-InformationCoding.md#8-when-rounding-errors-matter),
[data types](../Transcripts/02-InformationCoding.md#10-data-types-and-bitwise-operations) and
[character coding](../Transcripts/02-InformationCoding.md#9-character-coding) from session 2.